### 1. Install Ultralytics and Kaggle API
First, we need to install the `ultralytics` package to access YOLO11 models and the Kaggle CLI to download the dataset.

### Replace YOLO11 Head with CustomSegmentationHead
Now we will load the YOLO11 model and swap its default head with your `CustomSegmentationHead`. We'll identify the channel dimensions from the backbone to initialize your head correctly.

### Integration of Custom Head with YOLO11
Since the previous attempt had an environment issue, let's redefine the imports and swap the head dynamically.

In [ ]:
# 1. Install required libraries
!pip install ultralytics kaggle onnx

# IMPORTANT: After this finishes, click 'Runtime' -> 'Restart session'
# to fix the PyTorch circular import error before proceeding.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 4.1 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class CustomSegmentationHead(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.segmentation = nn.ModuleList([
            nn.Conv2d(c, 1, kernel_size=1)
            for c in channels
        ])

    def forward(self, features):

        # Convert each feature map to 1 channel
        masks = [
            self.segmentation[i](features[i])
            for i in range(len(features))
        ]

        # Resize P4 and P5 to P3's size
        target_size = masks[0].shape[-2:]

        masks = [
            F.interpolate(
                mask,
                size=target_size,
                mode="bilinear",
                align_corners=False
            )
            for mask in masks
        ]

        # Combine the three predictions
        mask = masks[0] + masks[1] + masks[2]

        return mask

In [ ]:
from ultralytics import YOLO

# Load the standard YOLOv8 segmentation model
# We are using the stock model as per your request to not use the custom head.
model = YOLO('yolov8n-seg.pt')

print("Standard YOLOv8 segmentation model loaded successfully.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Standard YOLOv8 segmentation model loaded successfully.


### 2. Download and Extract the Dataset
Please make sure you have uploaded your `kaggle.json` API token to the root directory of this Colab session.

In [ ]:
import os
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    os.makedirs('/root/.kaggle', exist_ok=True)
    # If you haven't uploaded it manually, this will prompt you:
    if not os.path.exists('kaggle.json'):
        print("Please upload your kaggle.json file")
        files.upload()
    !cp kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

# Download the dataset
!kaggle datasets download -d farzadnekouei/pothole-image-segmentation-dataset
!unzip -q pothole-image-segmentation-dataset.zip -d dataset

Please upload your kaggle.json file


cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/farzadnekouei/pothole-image-segmentation-dataset
License(s): apache-2.0
100% 59.3M/59.3M [00:02<00:00, 21.6MB/s]



### 3. Initialize YOLO11 Segmentation Model
We will load the YOLO11n-seg (nano) model, which is optimized for speed and segmentation tasks.

In [ ]:



# Start training
# Note: Ensure the 'data' path in the extracted dataset matches the path in your .yaml file
# Typically the dataset will contain a data.yaml file
results = model.train(
    data='dataset/Pothole_Segmentation_YOLOv8/data.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    device=0 # Uses GPU
)

Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/Pothole_Segmentation_YOLOv8/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, o

### 4. Evaluate and Predict
After training, you can run inference on test images to see how the model segments potholes.

### Run Inference on the Downloaded Video
Now that we have a dashcam video, we can run the custom model on the video file to see the segmentation in action.

In [ ]:
import yt_dlp
import os

# Updated to a new, active dashcam video URL
video_url = 'https://www.youtube.com/watch?v=cAON1pnMtvs'

ydl_opts = {
    'format': 'best[ext=mp4]',
    'outtmpl': 'test_dashcam.mp4',
    'quiet': False,
    'no_warnings': False,
    'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
}

try:
    # Remove existing file if download failed previously
    if os.path.exists('test_dashcam.mp4'):
        os.remove('test_dashcam.mp4')

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        print(f"Attempting to download video from {video_url}...")
        ydl.download([video_url])
    print("\nDownload complete. Video saved as 'test_dashcam.mp4'")
except Exception as e:
    print(f"Download failed: {e}")

Attempting to download video from https://www.youtube.com/watch?v=cAON1pnMtvs...
[youtube] Extracting URL: https://www.youtube.com/watch?v=cAON1pnMtvs
[youtube] cAON1pnMtvs: Downloading webpage


[youtube] cAON1pnMtvs: Downloading visionos player API JSON


ERROR: [youtube] cAON1pnMtvs: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


Download failed: ERROR: [youtube] cAON1pnMtvs: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


In [ ]:
import glob
import os
from ultralytics import YOLO

# Path to the sample video included in the dataset
sample_video_path = '/content/dataset/Pothole_Segmentation_YOLOv8/sample_video.mp4'

# Ensure we use the best weights from the training run
model_path = '/content/runs/segment/train/weights/best.pt'
if os.path.exists(model_path):
    inference_model = YOLO(model_path)
else:
    print("Training weights not found, using base model.")
    inference_model = YOLO('yolov8n-seg.pt')

# Run prediction on the dataset's sample video
if os.path.exists(sample_video_path):
    print(f"Running inference on {sample_video_path}...")
    results = inference_model.predict(
        source=sample_video_path,
        save=True,
        conf=0.25
    )

    # Locate the saved output
    predict_dirs = glob.glob('runs/segment/predict*/')
    if predict_dirs:
        latest_dir = max(predict_dirs, key=os.path.getmtime)
        print(f"Processed video saved to: {latest_dir}")
else:
    print(f"Sample video not found at {sample_video_path}. Please check your dataset extraction.")

Running inference on /content/dataset/Pothole_Segmentation_YOLOv8/sample_video.mp4...

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) /content/dataset/Pothole_Segmentation_YOLOv8/sample_video.mp4: 384x640 2 Potholes, 7.3ms
video 1/1 (frame 2/375) /content/dataset/Pothole_Segmentation_YOLOv8/sample_video.mp4: 384x640 3 Potholes, 18.3ms
video 1/1 (frame 3/375) /content/dataset/Pothole_Segmentation_YOLOv8/sample_video.mp4: 384x640 4 Potholes, 9.7ms
video 1/1 (frame 4/375) /content/

In [ ]:
pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 22.2 MB/s eta 0:00:00


In [ ]:
import inspect

print(inspect.signature(model.forward))

(*input: Any) -> None


In [ ]:
import torch

# Create a dummy input tensor for the model's expected input shape.
# YOLOv8 models typically take (batch_size, channels, image_height, image_width).
# From the training configuration, `imgsz=640`, so image_height and image_width are 640.
# We'll use a batch size of 1 for the dummy input to define the base shape.
dummy_input = torch.randn(1, 3, 640, 640)



# Define the dynamic_shapes for inputs ONLY, as required by torch.export when dynamo=True.
# The key for the input tensor in dynamic_shapes must match the argument name
# in the model's 'forward' method (typically 'x' for YOLO models).
# The 'input_names' parameter in torch.onnx.export will handle the ONNX node naming.
dynamic_input_shapes = (
    {
        0: torch.export.Dim("batch_size"),

    },
)

# Define output names for clarity in the ONNX graph
output_names_list = ["detection", "prototype"]


# disable batch normalizagtion
model.model.eval()

torch.onnx.export(
    model=model.model,
    args=dummy_input,  # Provide a sample input tensor
    f="model.onnx",
    opset_version=17,
    input_names=["input"], # This is the ONNX graph input name
    output_names=output_names_list,
    export_params=True,
    dynamo=True,
    dynamic_shapes=dynamic_input_shapes # Use the refined dynamic_input_shapes
)


W0908 21:14:29.091000 1984 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SegmentationModel([...]` with `torch.export.export(..., strict=False)`...


/usr/lib/python3.13/contextlib.py:148: UserWarning: The tensor attributes self.model.22.anchors, self.model.22.strides were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `SegmentationModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/BaseConverter.h:64: adapter_lookup: Ass

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[batch_size,3,640,640]>
            ),
            outputs=(
                %"detection"<FLOAT,[batch_size,37,8400]>,
                %"prototype"<FLOAT,[batch_size,32,160,160]>,
                %"cat_13"<FLOAT,[batch_size,64,8400]>,
                %"cat_14"<FLOAT,[batch_size,1,8400]>,
                %"silu_34"<FLOAT,[batch_size,64,80,80]>,
                %"silu_39"<FLOAT,[batch_size,128,40,40]>,
                %"silu_44"<FLOAT,[batch_size,256,20,20]>,
                %"cat_15"<FLOAT,[batch_size,32,8400]>,
                %"silu_65_alias_8"<FLOAT,[batch_size,32,160,160]>
            ),
            initializers=(
                %"model.0.conv.we

In [ ]:
import onnx

onnx.checker.check_model("model.onnx")